#Building Dataset


In [ ]:
!pip install ogb pyg-lib torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-2.3.0+cu121.html
!pip install torch-geometric


In [ ]:
# import gc

import torch
import torch.nn.functional as F
import torch.nn.functional as F
from torch.optim.lr_scheduler import ReduceLROnPlateau

import torch_geometric.transforms as T
from torch_geometric.data import NeighborSampler
from torch_geometric.nn import SAGEConv, GCNConv
from torch_geometric import utils, loader
from torch_geometric.utils import subgraph

from torch_geometric.nn import (
    Aggregation,
    MaxAggregation,
    MeanAggregation,
    MultiAggregation,
    SoftmaxAggregation,
    StdAggregation,
    SumAggregation,
    VarAggregation,
    LSTMAggregation
)

# importing obg datatset
#from ogb.nodeproppred import PygNodePropPredDataset
#from ogb.nodeproppred import PygNodePropPredDataset, Evaluator

from pandas.core.common import flatten
# import seaborn as sns
import matplotlib.pyplot as plt
# sns.set(rc={'figure.figsize':(16.7,8.27)})
# sns.set_theme(style="ticks")
from collections import Counter


from tqdm import tqdm
import collections
import os.path as osp
import pandas as pd
import numpy as np
import collections
# from pandas.core.common import flatten
# from scipy.special import softmax

import random

# Setting the seed
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed(42)
# torch.backends.cudnn.deterministic = True


In [ ]:
import pandas as pd

df = pd.read_parquet(
    "/content/sample_data/multimodal_embeddings.parquet"
)


In [ ]:
image_cols = [c for c in df.columns if c.startswith("img_")]


In [ ]:
text_cols = [c for c in df.columns if c.startswith("txt_")]


In [ ]:
from sklearn.neighbors import kneighbors_graph
import numpy as np
import torch

adj = kneighbors_graph(
    X,
    n_neighbors=5,
    mode="connectivity",
    include_self=True
)

edge_index = torch.tensor(
    np.array(adj.nonzero()),
    dtype=torch.long
)


In [ ]:
from torch_geometric.data import Data

data = Data(
    x=torch.tensor(
        X,
        dtype=torch.float
    ),
    edge_index=edge_index,
    y=torch.tensor(
        y,
        dtype=torch.long
    )
)


In [ ]:
print(data)


In [ ]:
# Define the indices of nodes to include in your subset
subset_indices = torch.arange(0, 5189)  # For example, first 10,000 nodes


# Extract the subgraph corresponding to the subset
subset_edge_index, edge_attr, edge_mask = subgraph(subset_indices, data.edge_index, None, relabel_nodes=True, num_nodes=data.num_nodes, return_edge_mask=True)


# Adjust node features and labels for the subset
subset_features = data.x[subset_indices]
subset_labels = data.y[subset_indices]

# Create a new graph object for the subset
subset_graph = data.__class__()
subset_graph.edge_index = subset_edge_index
subset_graph.x = subset_features
subset_graph.y = subset_labels


In [ ]:
# Safely infer num_features and num_classes from subset_graph
num_features = subset_graph.x.size(1)
if subset_graph.y.dim() == 1:
    num_classes = int(subset_graph.y.max().item()) + 1
elif subset_graph.y.dim() == 2:
    num_classes = subset_graph.y.size(1)

subset_graph.num_features = num_features
subset_graph.num_classes = num_classes
